# 🔍 TruthLens - BiLSTM Training on Google Colab

**Team Logic Lords | Heritage Institute of Technology**

---

## 📋 Instructions:
1. **Enable GPU**: Runtime → Change runtime type → GPU → Save
2. **Run cells in order** (Shift+Enter)
3. **Upload Fake.csv and True.csv** when prompted
4. **Wait 10-15 minutes** for training
5. **Download trained model** at the end

---

## 📦 Step 1: Install Dependencies

In [ ]:
!pip install -q pandas numpy scikit-learn nltk tensorflow

import nltk
nltk.download('stopwords', quiet=True)
nltk.download('punkt', quiet=True)

print("✅ Dependencies installed successfully!")

## 📁 Step 2: Upload Dataset Files

**Upload `Fake.csv` and `True.csv` from ISOT dataset**

In [ ]:
from google.colab import files
import os

print("📁 Please upload Fake.csv and True.csv")
print("(Click 'Choose Files' button below)\n")

uploaded = files.upload()

if 'Fake.csv' in uploaded and 'True.csv' in uploaded:
    print("\n✅ Both files uploaded successfully!")
    print(f"   Fake.csv: {len(uploaded['Fake.csv'])} bytes")
    print(f"   True.csv: {len(uploaded['True.csv'])} bytes")
else:
    print("\n❌ ERROR: Please upload both Fake.csv and True.csv")

## 🔧 Step 3: Define Preprocessing Functions

In [ ]:
import re
import pandas as pd
import numpy as np
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

STEMMER = PorterStemmer()
STOP_WORDS = set(stopwords.words("english"))

def clean_text(text: str) -> str:
    """Clean and preprocess text"""
    text = str(text).lower()
    text = re.sub(r"[^a-z\s]", " ", text)
    tokens = text.split()
    tokens = [STEMMER.stem(t) for t in tokens if t not in STOP_WORDS and len(t) > 2]
    return " ".join(tokens)

def load_isot_dataset(fake_path="Fake.csv", true_path="True.csv"):
    """Load and merge ISOT dataset"""
    print("📊 Loading dataset...")
    fake = pd.read_csv(fake_path)
    true = pd.read_csv(true_path)
    
    fake["label"] = 1  # 1 = FAKE
    true["label"] = 0  # 0 = REAL
    
    df = pd.concat([fake, true], ignore_index=True).sample(frac=1, random_state=42)
    df["content"] = df.get("title", "").fillna("") + " " + df.get("text", "").fillna("")
    
    print(f"   Cleaning {len(df)} articles (this may take 2-3 minutes)...")
    df["content"] = df["content"].apply(clean_text)
    
    return df

print("✅ Preprocessing functions defined!")

## 📊 Step 4: Load and Prepare Data

In [ ]:
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Load dataset
df = load_isot_dataset()

print(f"\n📈 Dataset Statistics:")
print(f"   Total samples: {len(df)}")
print(f"   FAKE news: {sum(df['label']==1)} ({sum(df['label']==1)/len(df)*100:.1f}%)")
print(f"   REAL news: {sum(df['label']==0)} ({sum(df['label']==0)/len(df)*100:.1f}%)")

# Hyperparameters
MAX_VOCAB = 35000
MAX_LEN = 256
BATCH_SIZE = 128
EPOCHS = 10

print(f"\n🔧 Tokenizing text...")
print(f"   Max vocabulary: {MAX_VOCAB}")
print(f"   Max sequence length: {MAX_LEN}")

tok = Tokenizer(num_words=MAX_VOCAB, oov_token="<OOV>")
tok.fit_on_texts(df["content"])
seqs = tok.texts_to_sequences(df["content"])
X = pad_sequences(seqs, maxlen=MAX_LEN, padding="post", truncating="post")
y = df["label"].values

print(f"   Vocabulary size: {len(tok.word_index)}")
print(f"   Sequence shape: {X.shape}")

# Split data
print(f"\n📦 Splitting data...")
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train, test_size=0.1, random_state=42, stratify=y_train
)

print(f"   Train: {len(X_train):,} samples")
print(f"   Val:   {len(X_val):,} samples")
print(f"   Test:  {len(X_test):,} samples")

print("\n✅ Data preparation complete!")

## 🧠 Step 5: Build BiLSTM Model Architecture

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (Input, Embedding, Conv1D, MaxPooling1D,
                                     Bidirectional, LSTM, Dense, Dropout, Layer)
import tensorflow.keras.backend as K

class AttentionLayer(Layer):
    """Custom Attention Mechanism"""
    def __init__(self, units=32, **kwargs):
        super().__init__(**kwargs)
        self.units = units
    
    def build(self, input_shape):
        self.W = self.add_weight(shape=(input_shape[-1], self.units),
                                 initializer="glorot_uniform", trainable=True)
        self.b = self.add_weight(shape=(self.units,),
                                 initializer="zeros", trainable=True)
        self.u = self.add_weight(shape=(self.units, 1),
                                 initializer="glorot_uniform", trainable=True)
        super().build(input_shape)
    
    def call(self, H):
        score = K.tanh(K.dot(H, self.W) + self.b)
        score = K.dot(score, self.u)
        alpha = K.softmax(score, axis=1)
        context = K.sum(H * alpha, axis=1)
        return context, alpha
    
    def get_config(self):
        cfg = super().get_config()
        cfg.update({"units": self.units})
        return cfg

def build_bilstm_model(vocab_size):
    """Build Attention BiLSTM Model"""
    inp = Input(shape=(MAX_LEN,), name='input')
    x = Embedding(vocab_size, 64, name='embedding')(inp)
    x = Conv1D(64, 5, activation="relu", padding="same", name='conv1d')(x)
    x = MaxPooling1D(pool_size=2, name='maxpool')(x)
    x = Bidirectional(LSTM(64, return_sequences=True), name='bilstm')(x)
    context, _ = AttentionLayer(32, name='attention')(x)
    x = Dense(64, activation="relu", name='dense1')(context)
    x = Dropout(0.3, name='dropout1')(x)
    x = Dense(32, activation="relu", name='dense2')(x)
    x = Dropout(0.3, name='dropout2')(x)
    out = Dense(1, activation="sigmoid", name='output')(x)
    
    model = Model(inputs=inp, outputs=out, name='AttentionBiLSTM')
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )
    return model

print("🏗️ Building model...")
vocab_size = min(MAX_VOCAB, len(tok.word_index)) + 1
model = build_bilstm_model(vocab_size)

print("\n📐 Model Architecture:")
model.summary()

print(f"\n✅ Model built successfully!")
print(f"   Total parameters: {model.count_params():,}")

## 🚀 Step 6: Train Model

**⏱️ This will take 5-10 minutes with GPU (30+ minutes without GPU)**

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
import os

os.makedirs('saved_model', exist_ok=True)

print("🚀 Starting training...")
print("⏱️  Estimated time: 5-10 minutes (with GPU)")
print("\n" + "="*70)

callbacks = [
    EarlyStopping(
        monitor="val_loss", 
        patience=3, 
        restore_best_weights=True, 
        verbose=1
    ),
    ModelCheckpoint(
        "saved_model/attention_bilstm_model.keras", 
        save_best_only=True, 
        verbose=1
    )
]

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=callbacks,
    verbose=1
)

print("\n" + "="*70)
print("✅ Training complete!")

## 📊 Step 7: Evaluate Model Performance

In [ ]:
print("📊 Evaluating model on test set...\n")

loss, acc = model.evaluate(X_test, y_test, verbose=0)

print("="*70)
print("🎯 TEST RESULTS")
print("="*70)
print(f"Loss:     {loss:.4f}")
print(f"Accuracy: {acc:.4f} ({acc*100:.2f}%)")
print("="*70)

# Test predictions on samples
print("\n🧪 Testing predictions on random samples...\n")

fake_idx = np.where(y_test == 1)[0][:5]
real_idx = np.where(y_test == 0)[0][:5]

print("FAKE News Samples (should predict > 0.5):")
for i, idx in enumerate(fake_idx, 1):
    pred = model.predict(X_test[idx:idx+1], verbose=0)[0][0]
    verdict = "FAKE" if pred > 0.5 else "REAL"
    emoji = "✅" if verdict == "FAKE" else "❌"
    print(f"  {i}. {emoji} Prediction: {pred:.4f} → {verdict}")

print("\nREAL News Samples (should predict < 0.5):")
for i, idx in enumerate(real_idx, 1):
    pred = model.predict(X_test[idx:idx+1], verbose=0)[0][0]
    verdict = "FAKE" if pred > 0.5 else "REAL"
    emoji = "✅" if verdict == "REAL" else "❌"
    print(f"  {i}. {emoji} Prediction: {pred:.4f} → {verdict}")

print("\n✅ Evaluation complete!")

## 🧪 Step 8: Test with Custom Examples

In [ ]:
def predict_text(text):
    """Predict if text is FAKE or REAL"""
    cleaned = clean_text(text)
    seq = tok.texts_to_sequences([cleaned])
    padded = pad_sequences(seq, maxlen=MAX_LEN, padding="post")
    pred = model.predict(padded, verbose=0)[0][0]
    verdict = "FAKE" if pred > 0.5 else "REAL"
    return verdict, pred

print("🧪 Testing with custom examples:\n")
print("="*70)

test_cases = [
    ("FAKE", "SHOCKING: Secret government documents reveal alien technology in smartphones!"),
    ("REAL", "President announces new economic policy during press conference today"),
    ("FAKE", "BREAKING NEWS: Whistleblower exposes massive conspiracy! Click here!"),
    ("REAL", "According to Reuters, the stock market showed gains in morning trading"),
    ("FAKE", "Anonymous sources confirm secret documents prove government cover-up"),
    ("REAL", "The Federal Reserve announced interest rate decision following committee meeting")
]

correct = 0
for expected, text in test_cases:
    verdict, confidence = predict_text(text)
    is_correct = verdict == expected
    correct += is_correct
    emoji = "✅" if is_correct else "❌"
    
    print(f"{emoji} Expected: {expected} | Predicted: {verdict} ({confidence:.4f})")
    print(f"   {text[:65]}...")
    print()

print("="*70)
print(f"Custom Test Accuracy: {correct}/{len(test_cases)} ({correct/len(test_cases)*100:.1f}%)")
print("="*70)

## 💾 Step 9: Save Model and Tokenizer

In [ ]:
import pickle

print("💾 Saving model and tokenizer...\n")

# Save model
model.save("saved_model/attention_bilstm_model.keras")
print("✅ Model saved: saved_model/attention_bilstm_model.keras")

# Save tokenizer
with open("saved_model/tokenizer.pkl", "wb") as f:
    pickle.dump(tok, f)
print("✅ Tokenizer saved: saved_model/tokenizer.pkl")

print("\n📦 Files ready for download!")

## ⬇️ Step 10: Download Trained Files

In [ ]:
from google.colab import files

print("⬇️ Downloading trained model files...\n")

print("Downloading: attention_bilstm_model.keras")
files.download("saved_model/attention_bilstm_model.keras")

print("Downloading: tokenizer.pkl")
files.download("saved_model/tokenizer.pkl")

print("\n" + "="*70)
print("✅ DOWNLOAD COMPLETE!")
print("="*70)
print("\n📝 Next Steps:")
print("  1. Copy downloaded files to your project folder:")
print("     Fake News/saved_model/")
print("  2. Replace the old files with these new ones")
print("  3. Test with: python quick_test.py 'Your article'")
print("  4. BiLSTM will now work properly!")
print("\n🎉 Training successful!")

## 📈 Step 11: Visualize Training History (Optional)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(14, 5))

# Accuracy plot
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Train Accuracy', marker='o')
plt.plot(history.history['val_accuracy'], label='Val Accuracy', marker='s')
plt.title('Model Accuracy Over Epochs', fontsize=14, fontweight='bold')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True, alpha=0.3)

# Loss plot
plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Train Loss', marker='o')
plt.plot(history.history['val_loss'], label='Val Loss', marker='s')
plt.title('Model Loss Over Epochs', fontsize=14, fontweight='bold')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n📊 Training Summary:")
print(f"   Final Train Accuracy: {history.history['accuracy'][-1]*100:.2f}%")
print(f"   Final Val Accuracy:   {history.history['val_accuracy'][-1]*100:.2f}%")
print(f"   Test Accuracy:        {acc*100:.2f}%")
print(f"   Total Epochs:         {len(history.history['accuracy'])}")

---

## 🎉 Training Complete!

### ✅ What You've Accomplished:
- ✅ Trained Attention BiLSTM model on 44,898 articles
- ✅ Achieved high accuracy on fake news detection
- ✅ Downloaded trained model files

### 📝 Next Steps:
1. Copy `attention_bilstm_model.keras` and `tokenizer.pkl` to your project
2. Place them in `Fake News/saved_model/` folder
3. Run `python quick_test.py "Your article text"`
4. BiLSTM will now detect fake news properly!

---

**Built with ❤️ by Team Logic Lords**

Heritage Institute of Technology | 2025-2026